# CRM сегменты: визуализации для бизнеса

Отдельная тетрадка с двумя правками:
- динамика ФП/СФП по месяцам с выделением границ годов полупрозрачными пунктирными линиями;
- круговая диаграмма уникальных ИНН по сегментам.

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


def add_year_separators(ax, dt_index):
    years = sorted(dt_index.year.unique())
    for year in years[1:]:
        boundary = pd.Timestamp(f"{year}-01-01")
        ax.axvline(boundary, color="gray", linestyle="--", linewidth=1.2, alpha=0.4)


In [ ]:
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df["inn"] = df["X_INN"].apply(normalize_inn)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
if "VAL" in df.columns:
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()

df["TYPE_FP"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
df = df[df["segment"].notna()].copy()

df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
df["year_month"] = df["dt"].dt.to_period("M")

print(f"Готово к анализу: {len(df):,} строк")
print(f"Уникальных ИНН: {df['inn'].nunique():,}")

## Уникальные ИНН по сегментам: круговая диаграмма

In [ ]:
total_inn = df["inn"].nunique()
inn_by_seg = df.groupby("segment")["inn"].nunique().reindex(SEG_ORDER, fill_value=0)
inn_share = (inn_by_seg / total_inn * 100).round(1)

tbl_inn = pd.DataFrame({
    "Сегмент": SEG_ORDER,
    "Уникальных ИНН": [inn_by_seg[s] for s in SEG_ORDER],
    "Доля, %": [inn_share[s] for s in SEG_ORDER],
})
display(tbl_inn.style.hide(axis="index"))

# Более презентационный стиль: "donut" + акцент крупнейшего сегмента + тень.
colors = ["#2E7D32", "#66BB6A", "#A5D6A7"]
explode = [0.06 if v == inn_by_seg.max() else 0.02 for v in inn_by_seg.values]

fig, ax = plt.subplots(figsize=(8, 7))
wedges, texts, autotexts = ax.pie(
    inn_by_seg.values,
    labels=SEG_ORDER,
    autopct=lambda p: f"{p:.1f}%",
    startangle=140,
    explode=explode,
    shadow=True,
    colors=colors,
    wedgeprops={"edgecolor": "white", "linewidth": 1.2, "width": 0.52},
    pctdistance=0.78,
    labeldistance=1.05,
    textprops={"fontsize": 11, "fontweight": "bold"},
)

# Центр donut: общий итог и подпись.
ax.text(0, 0.06, f"{total_inn:,}", ha="center", va="center", fontsize=16, fontweight="bold")
ax.text(0, -0.10, "уникальных ИНН", ha="center", va="center", fontsize=10, color="dimgray")

# Доп. легенда с абсолютными значениями для слайда.
legend_labels = [f"{seg}: {inn_by_seg[seg]:,} ({inn_share[seg]:.1f}%)" for seg in SEG_ORDER]
ax.legend(wedges, legend_labels, title="Сегменты", loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

ax.set_title("Распределение уникальных ИНН по сегментам", fontsize=14, fontweight="bold", pad=14)
ax.axis("equal")
plt.tight_layout()
plt.show()

## Динамика ФП/СФП с выделением границ годов

In [ ]:
def plot_monthly_dynamics(data, fp_type, title):
    subset = data[data["TYPE_FP"] == fp_type].copy()
    pivot = subset.groupby(["year_month", "segment"]).size().unstack(fill_value=0)
    pivot = pivot.reindex(columns=SEG_ORDER, fill_value=0)
    pivot.index = pivot.index.to_timestamp()

    # Полный календарь месяцев за период: оставляем даже месяцы с нулевыми значениями.
    full_months = pd.date_range(
        DATE_FROM.to_period("M").to_timestamp(),
        DATE_TO.to_period("M").to_timestamp(),
        freq="MS",
    )
    pivot = pivot.reindex(index=full_months, fill_value=0)

    fig, ax = plt.subplots(figsize=(14, 5))
    pivot.plot(ax=ax, marker="o", markersize=3, linewidth=1.2)
    add_year_separators(ax, pivot.index)

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Месяц")
    ax.set_ylabel(f"Количество {fp_type}")
    ax.legend(title="Сегмент")

    # Показываем все месяцы на оси X.
    ax.set_xticks(full_months)
    ax.set_xticklabels(full_months.strftime("%Y-%m"), rotation=45, ha="right")

    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()


plot_monthly_dynamics(df, "ФП", "Количество выявленных ФП по месяцам (2023-2025)")
plot_monthly_dynamics(df, "СФП", "Количество выявленных СФП по месяцам (2023-2025)")

In [ ]:
# Логика дефолта как в analysis_default_logic_eda.ipynb (ДМУКП)
DEFAULT_RULES = {
    "ККБ": {
        "FP": {"13", "13.1", "15.1.3.1"},
        "SFP": {"15.2.1", "15.2.2", "15.2.1.1", "15.2.1.2"},
    },
    "МСБ": {
        "FP": {"13", "15.1.3"},
        "SFP": {"15.2", "15.2.1.1"},
    },
    "МкБ": {
        "FP": {"13", "15.1.2"},
        "SFP": {"15.2", "15.2У", "15.2.1У"},
    },
}

MIN_OBS_MONTHS = 3
MIN_OBS_DATE = DATE_FROM + pd.DateOffset(months=MIN_OBS_MONTHS)

# 1) Определяем дефолтные события по сегментным правилам
parts = []
for seg in SEG_ORDER:
    rules = DEFAULT_RULES[seg]
    seg_df = df[df["segment"] == seg]
    mask_fp = (seg_df["fp_num"].isin(rules["FP"])) & (seg_df["TYPE_FP"] == "ФП")
    mask_sfp = (seg_df["fp_num"].isin(rules["SFP"])) & (seg_df["TYPE_FP"] == "СФП")
    parts.append(seg_df[mask_fp | mask_sfp])

df_def_events = pd.concat(parts, ignore_index=True)

defaults = (
    df_def_events
    .groupby(["inn", "segment"], as_index=False)
    .agg(default_date=("dt", "min"))
)
defaults["default_flag"] = 1

# 2) Реестр компаний + дефолт
companies = df[["inn", "segment"]].drop_duplicates()
companies = companies.merge(
    defaults[["inn", "segment", "default_flag", "default_date"]],
    on=["inn", "segment"], how="left"
)
companies["default_flag"] = companies["default_flag"].fillna(0).astype(int)

# 3) Минимальное окно наблюдения (как в original notebook)
early_default_inns = companies[
    (companies["default_flag"] == 1) & (companies["default_date"] < MIN_OBS_DATE)
]["inn"].unique()
companies = companies[~companies["inn"].isin(early_default_inns)].copy()

# 4) Помесячная / поквартальная динамика дефолтов (original style)
def_companies = companies[companies["default_flag"] == 1].copy()
def_companies["def_month"] = def_companies["default_date"].dt.to_period("M")
def_companies["def_quarter"] = def_companies["default_date"].dt.to_period("Q")

monthly = def_companies.groupby(["def_month", "segment"]).size().unstack(fill_value=0)
monthly = monthly[[s for s in SEG_ORDER if s in monthly.columns]]
monthly["Итого"] = monthly.sum(axis=1)
monthly.index = monthly.index.astype(str)

quarterly = def_companies.groupby(["def_quarter", "segment"]).size().unstack(fill_value=0)
quarterly = quarterly[[s for s in SEG_ORDER if s in quarterly.columns]]
quarterly["Итого"] = quarterly.sum(axis=1)
quarterly.index = quarterly.index.astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
colors = {"МкБ": "#4472C4", "МСБ": "#ED7D31", "ККБ": "#70AD47"}

for seg in SEG_ORDER:
    if seg in monthly.columns:
        axes[0].plot(monthly.index, monthly[seg], marker="o", markersize=3,
                     label=seg, color=colors[seg], linewidth=1.5)

# Добавляем только пунктирные полупрозрачные границы годов
month_dt = pd.to_datetime(monthly.index.astype(str))
years = sorted(month_dt.year.unique())
for year in years[1:]:
    year_start = f"{year}-01"
    if year_start in monthly.index:
        x_pos = monthly.index.get_loc(year_start)
        axes[0].axvline(x=x_pos, color="gray", linestyle="--", linewidth=1.2, alpha=0.4)

axes[0].set_title("Новые дефолты по месяцам", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Количество ИНН")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45, labelsize=7)

seg_cols = [s for s in SEG_ORDER if s in quarterly.columns]
quarterly[seg_cols].plot(kind="bar", stacked=False, ax=axes[1],
                         color=[colors[s] for s in seg_cols], edgecolor="white", width=0.8)
axes[1].set_title("Новые дефолты по кварталам", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Количество ИНН")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Компаний с дефолтом после фильтра окна: {len(def_companies):,}")